# Data Preprocessing Pipeline

This notebook handles the complete data preprocessing pipeline for the Telco Customer Churn dataset.

In [1]:
# Install required packages
# %pip install pandas scikit-learn pyarrow

# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from pathlib import Path

In [2]:
# Configuration
DATA_PATH = "../data/raw/telco_customer_churn.csv"
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

In [3]:
# Load dataset
print("Loading dataset...")
data = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

Loading dataset...
Dataset shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Data inspection
print("=== Data Inspection ===")
print(f"\nShape: {data.shape}")
print(f"\nData types:\n{data.dtypes}")
print(f"\nMissing values:\n{data.isnull().sum()}")
print(f"\nStatistical summary:")
print(data.describe())

=== Data Inspection ===

Shape: (7043, 21)

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies  

In [5]:
# Handle missing values and data conversion
print("=== Handling Missing Values and Data Conversion ===")

# Convert TotalCharges from string to numeric
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

print(f"Rows with missing values: {data.isnull().any(axis=1).sum()}")

# Remove rows with any missing values
data_clean = data.dropna()
print(f"Dataset shape after cleaning: {data_clean.shape}")
print(f"Removed {data.shape[0] - data_clean.shape[0]} rows")

=== Handling Missing Values and Data Conversion ===
Rows with missing values: 11
Dataset shape after cleaning: (7032, 21)
Removed 11 rows


In [6]:
# Define feature categories
numerical_columns = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

categorical_columns = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

identifier_column = 'customerID'
target_column = 'Churn'

print(f"Numerical features: {len(numerical_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(f"Identifier: {identifier_column}")
print(f"Target: {target_column}")

Numerical features: 4
Categorical features: 15
Identifier: customerID
Target: Churn


In [7]:
# Separate features and target
print("=== Separating Features and Target ===")

features = data_clean.drop(columns=[target_column])

# Encode target to binary (No=0, Yes=1) so downstream metrics work without pos_label
labels = (data_clean[target_column] == 'Yes').astype('int64')

print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Label distribution:\n{labels.value_counts()}")

=== Separating Features and Target ===
Features shape: (7032, 20)
Labels shape: (7032,)
Label distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [8]:
# Train/test split
print("=== Train/Test Split ===")

from typing import cast

_splits = train_test_split(
    features, labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
X_train: pd.DataFrame = cast(pd.DataFrame, _splits[0])
X_test: pd.DataFrame = cast(pd.DataFrame, _splits[1])
y_train: pd.Series = cast(pd.Series, _splits[2])
y_test: pd.Series = cast(pd.Series, _splits[3])

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test labels: {y_test.shape}")

=== Train/Test Split ===
Training set: (5625, 20)
Test set: (1407, 20)
Training labels: (5625,)
Test labels: (1407,)


In [9]:
# Scale numerical features
print("=== Scaling Numerical Features ===")

X_train = cast(pd.DataFrame, X_train)
X_test = cast(pd.DataFrame, X_test)

scaler = StandardScaler()

# Fit on training data
X_train_numerical = X_train[numerical_columns]
scaler.fit(X_train_numerical)

# Transform training data
X_train_scaled = scaler.transform(X_train_numerical)
X_train_scaled_df = pd.DataFrame(
    data=X_train_scaled,
    index=X_train.index,
    columns=numerical_columns
)

# Transform test data
X_test_numerical = X_test[numerical_columns]
X_test_scaled = scaler.transform(X_test_numerical)
X_test_scaled_df = pd.DataFrame(
    data=X_test_scaled,
    index=X_test.index,
    columns=numerical_columns
)

print(f"Scaled training data shape: {X_train_scaled_df.shape}")
print(f"Scaled test data shape: {X_test_scaled_df.shape}")

=== Scaling Numerical Features ===
Scaled training data shape: (5625, 4)
Scaled test data shape: (1407, 4)


In [10]:
# Encode categorical features
print("=== Encoding Categorical Features ===")

X_train = cast(pd.DataFrame, X_train)
X_test = cast(pd.DataFrame, X_test)

encoder = OneHotEncoder(sparse_output=False)

# Fit on training data
X_train_categorical = X_train[categorical_columns]
encoder.fit(X_train_categorical)

# Transform training data
X_train_encoded = encoder.transform(X_train_categorical)
X_train_encoded_df = pd.DataFrame(
    data=X_train_encoded,
    index=X_train.index,
    columns=encoder.get_feature_names_out()
)

# Transform test data
X_test_categorical = X_test[categorical_columns]
X_test_encoded = encoder.transform(X_test_categorical)
X_test_encoded_df = pd.DataFrame(
    data=X_test_encoded,
    index=X_test.index,
    columns=encoder.get_feature_names_out()
)

print(f"Encoded training data shape: {X_train_encoded_df.shape}")
print(f"Encoded test data shape: {X_test_encoded_df.shape}")
print(f"Encoded columns: {list(encoder.get_feature_names_out())}")

=== Encoding Categorical Features ===
Encoded training data shape: (5625, 41)
Encoded test data shape: (1407, 41)
Encoded columns: ['gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_No', 'PaperlessBilling_Y

In [11]:
# Combine processed features
print("=== Combining Processed Features ===")

X_train = cast(pd.DataFrame, X_train)
X_test = cast(pd.DataFrame, X_test)

# Combine for training set
X_train_processed = pd.concat([
    X_train[identifier_column],
    X_train_scaled_df,
    X_train_encoded_df
], axis=1)

# Combine for test set
X_test_processed = pd.concat([
    X_test[identifier_column],
    X_test_scaled_df,
    X_test_encoded_df
], axis=1)

print(f"Processed training data shape: {X_train_processed.shape}")
print(f"Processed test data shape: {X_test_processed.shape}")

=== Combining Processed Features ===
Processed training data shape: (5625, 46)
Processed test data shape: (1407, 46)


In [12]:
# Save processed data
print("=== Saving Processed Data ===")

train_path = OUTPUT_DIR / "train.parquet"
test_path = OUTPUT_DIR / "test.parquet"
train_labels_path = OUTPUT_DIR / "train_labels.parquet"
test_labels_path = OUTPUT_DIR / "test_labels.parquet"

X_train_processed.to_parquet(train_path)
X_test_processed.to_parquet(test_path)
y_train.to_frame().to_parquet(train_labels_path)
y_test.to_frame().to_parquet(test_labels_path)

print(f"Saved training data: {train_path}")
print(f"Saved test data: {test_path}")
print(f"Saved training labels: {train_labels_path}")
print(f"Saved test labels: {test_labels_path}")

=== Saving Processed Data ===
Saved training data: ../data/processed/train.parquet
Saved test data: ../data/processed/test.parquet
Saved training labels: ../data/processed/train_labels.parquet
Saved test labels: ../data/processed/test_labels.parquet


In [13]:
# Save preprocessing objects
print("=== Saving Preprocessing Objects ===")

import pickle

scaler_path = OUTPUT_DIR / "scaler.pkl"
encoder_path = OUTPUT_DIR / "encoder.pkl"

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

with open(encoder_path, 'wb') as f:
    pickle.dump(encoder, f)

print(f"Saved scaler: {scaler_path}")
print(f"Saved encoder: {encoder_path}")

=== Saving Preprocessing Objects ===
Saved scaler: ../data/processed/scaler.pkl
Saved encoder: ../data/processed/encoder.pkl


In [14]:
# Summary
print("=== Preprocessing Summary ===")
print(f"\nOriginal dataset: {data.shape}")
print(f"After cleaning: {data_clean.shape}")
print(f"Training set: {X_train_processed.shape}")
print(f"Test set: {X_test_processed.shape}")
print(f"\nNumerical features scaled: {len(numerical_columns)}")
print(f"Categorical features encoded: {len(categorical_columns)}")
print(f"Total features after preprocessing: {X_train_processed.shape[1] - 1}")  # -1 for customerID
print(f"\nChurn rate in training: {(y_train == 1).mean():.2%}")
print(f"Churn rate in test: {(y_test == 1).mean():.2%}")

=== Preprocessing Summary ===

Original dataset: (7043, 21)
After cleaning: (7032, 21)
Training set: (5625, 46)
Test set: (1407, 46)

Numerical features scaled: 4
Categorical features encoded: 15
Total features after preprocessing: 45

Churn rate in training: 26.58%
Churn rate in test: 26.58%
